<a href="https://colab.research.google.com/github/MateuszCzz/Osu-Beatmaps-Preformance-Data-Analyzer/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. imports, config

In [ ]:
# Dependencies
!pip install -q prettytable tqdm
!pip install ossapi==5.3.3
import csv, json, os, random, time, glob
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Optional
from google.colab import drive, files, output
from google.colab.output import eval_js
from ossapi import Ossapi
from tqdm.notebook import tqdm

# @markdown Get your API credentials from:
# @markdown [https://osu.ppy.sh/home/account/edit](https://osu.ppy.sh/home/account/edit) → "New OAuth Application"
# @markdown
# @markdown | Field | Description |
# @markdown |-------|-------------|
# @markdown | **Client ID** | Your app’s numeric ID (used to authenticate requests) |
# @markdown | **Client Secret** | Your app’s private key (keep this secret, don’t share it) |
osu_api_client_id     = 0   # @param {type:"integer"}
osu_api_client_secret = ""  # @param {type:"string"}

# Configuration
mode_chosen  = "osu"
ranking_type = "performance"
num_parts = 10

# @markdown Ranking position range, for users that will be considered in collecting data
rank_min = 100_000  # @param {type:"integer"}
rank_max = 200_000  # @param {type:"integer"}

# @markdown Filter scores by performance points (PP)
min_pp    = 100  # @param {type:"integer"}

# @markdown Where to save data in your Google Drive
folder_name = "osu_data_scrapper" # @param {type:"string"}
SAVE_DIR         = f"/content/drive/MyDrive/{folder_name}"
RESULT_DIR_USERS = f"{SAVE_DIR}/results_users"

# Score scraping params
SCORE_BATCH_SIZE = 1_000
MAX_WORKERS      = 2
SCORE_LIMIT      = 100

# Raw scan batch size (separate from score batching)
SCAN_BATCH_SIZE = 50

# Loading scores params
# @markdown Minimum number of scores on a beatmap (with same mods) to keep it
min_score_count = 5  # @param {type:"integer"}

# Beatmap dig params
CACHE_SAVE_INTERVAL = 100

# @markdown Maximum number of top player tags stored per beatmap
MAX_TAGS            = 4    # @param {type:"integer"}


_KEEP_MODS    = {"EZ", "HR", "DT", "HF"}
_REPLACE_MODS = {"NC": "DT"}

OUTPUT_FILENAME = "osu_beatmap_report"

# Init
drive.mount("/content/drive")
os.makedirs(SAVE_DIR,         exist_ok=True)
os.makedirs(RESULT_DIR_USERS, exist_ok=True)
RESULT_DIR_SCORES = f"{SAVE_DIR}/scores"
os.makedirs(RESULT_DIR_SCORES, exist_ok=True)
CACHE_FILE = os.path.join(SAVE_DIR, "beatmapset_cache.json")
api = Ossapi(osu_api_client_id, osu_api_client_secret)
output.clear()
print("✓ Ready")

# Data Models
@dataclass
class UserCompact:
    id:          int
    country:     str
    global_rank: int
    pp:          float

@dataclass
class Beatmap:
    id:                int
    hit_length:        int
    diff:              str
    difficulty_rating: float
    last_updated_year: int
    playcount:         int

@dataclass
class BeatmapsetCompact:
    id:      int
    creator: str
    title:   str

@dataclass
class Score:
    mods:       str
    pp:         float
    beatmap:    Beatmap
    beatmapset: BeatmapsetCompact
    perfect:    bool
    accuracy:   float
    max_combo:  int
    user_id:    int

@dataclass
class ScoreCompact:
    id:               int
    user_id:          int
    beatmap_id:       int
    artist:           str
    creator:          str
    title:            str
    difficulty_name:  str
    mods:             str
    pp:               float
    accuracy:         float
    combo:            int
    rating:           float
    bpm:              float
    length:           int
    ar:               float
    od:               float
    cs:               float
    hp:               float
    spinners:         int
    year:             int
    playcount:        int
    favourite_count:  int
    missless:         bool
    is_hidden:        bool

@dataclass
class BeatmapEntry:
    beatmap_id:       int
    mods:             str
    artist:           str
    creator:          str
    title:            str
    difficulty_name:  str
    rating:           float
    bpm:              float
    length:           int
    ar:               float
    od:               float
    cs:               float
    hp:               float
    spinners:         int
    year:             int
    playcount:        int
    favourite_count:  int
    combo:            int
    pp_avg:           float
    accuracy_avg:     float
    hidden_rate:      float
    missless_rate:    float
    amount_of_scores: int

@dataclass
class EnhancedBeatmapEntry(BeatmapEntry):
    genre:    Optional[str] = None
    language: Optional[str] = None
    tags:     Optional[str] = None  # comma-joined top-N tag names

# Shared Utilities

def retry_api_call(fn, *args, retries=10, base_delay=1, **kwargs):
    """Exponential back-off wrapper around any API call."""
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == retries - 1:
                raise RuntimeError(
                    f"API call failed after {retries} retries: {fn.__name__}"
                ) from e
            delay = base_delay * (2 ** attempt) + random.random()
            print(f"  ↻ {fn.__name__} error: {e}  — retry in {delay:.1f}s")
            time.sleep(delay)


def load_progress(name: str, default: int = 0) -> int:
    path = os.path.join(SAVE_DIR, name)
    if not os.path.exists(path):
        return default
    with open(path) as f:
        return int(f.read().strip())


def save_progress(name: str, value: int) -> None:
    with open(os.path.join(SAVE_DIR, name), "w") as f:
        f.write(str(value))


def save_users(filename: str, users: list[UserCompact]) -> None:
    """Saves users to a JSONL file, overwriting any existing file. Skips empty batches."""
    if not users:
        return
    path = os.path.join(RESULT_DIR_USERS, f"{filename}.jsonl")
    with open(path, "w") as f:
        for u in users:
            f.write(json.dumps(asdict(u)) + "\n")


def _parse_mods(raw_mods) -> tuple[str, bool]:
    """Return (mods_string, is_hidden). NM when no relevant mods remain."""

    # Ensure raw_mods is iterable
    if raw_mods is None:
        raw_mods = []
    elif not isinstance(raw_mods, (list, tuple)):
        raw_mods = [raw_mods]

    mods, is_hidden = [], False
    for mod in raw_mods:
        acronym = mod.acronym if hasattr(mod, "acronym") else str(mod)
        if acronym == "HD":
            is_hidden = True
        elif acronym in _REPLACE_MODS:
            mods.append(_REPLACE_MODS[acronym])
        elif acronym in _KEEP_MODS:
            mods.append(acronym)

    return ("NM" if not mods else "".join(sorted(mods))), is_hidden


def _parse_missless(stats) -> bool:
    if hasattr(stats, "miss"):
        return stats.miss == 0
    if isinstance(stats, dict):
        return stats.get("miss", 0) == 0
    return False


def convert_to_score_compact(score) -> ScoreCompact:
    bm  = score.beatmap
    bms = score.beatmapset

    mods_str, is_hidden = _parse_mods(getattr(score, "mods", []))

    last_updated = getattr(bm, "last_updated", None)
    year = last_updated.year if last_updated else 0

    return ScoreCompact(
        id              = int(getattr(score, "id", 0)),
        user_id         = int(getattr(score, "user_id", 0)),
        beatmap_id      = int(getattr(score, "beatmap_id", None) or bm.id),
        artist          = str(getattr(bms, "artist", "")),
        creator         = str(getattr(bms, "creator", "")),
        title           = str(getattr(bms, "title", "")),
        difficulty_name = str(getattr(bm, "version", "")),
        mods            = mods_str,
        pp              = float(getattr(score, "pp", 0.0)),
        accuracy        = float(getattr(score, "accuracy", 0.0)),
        combo           = int(getattr(score, "max_combo", 0)),
        rating          = float(getattr(bm, "difficulty_rating", 0.0)),
        bpm             = float(getattr(bm, "bpm", 0.0)),
        length          = int(getattr(bm, "total_length", 0)),
        ar              = float(getattr(bm, "ar", 0.0)),
        od              = float(getattr(bm, "accuracy", 0.0)),
        cs              = float(getattr(bm, "cs", 0.0)),
        hp              = float(getattr(bm, "drain", 0.0)),
        spinners        = int(getattr(bm, "count_spinners", 0)),
        year            = year,
        playcount       = int(getattr(bm, "playcount", 0)),
        favourite_count = int(getattr(bms, "favourite_count", 0)),
        missless        = _parse_missless(getattr(score, "statistics", {})),
        is_hidden       = is_hidden,
    )


def fetch_user_scores(user: UserCompact) -> list[ScoreCompact]:
    raw = retry_api_call(
        api.user_scores,
        user_id=user.id,
        type="best",
        include_fails=False,
        limit=SCORE_LIMIT,
        mode=mode_chosen,
    )
    return [
        convert_to_score_compact(s)
        for s in raw
        if getattr(s, "pp", 0.0) and s.pp >= min_pp
    ]

### API Credentials

*   **`api_client_id`**: Your osu! API client ID. You can obtain this from your osu! account settings: [https://osu.ppy.sh/home/account/api](https://osu.ppy.sh/home/account/api).
*   **`api_client_secret`**: Your osu! API client secret, also obtained from the same API settings page. Keep this confidential.


# 2. Acquiring User Information

Method 1 — Country rankings Cell (best option)

In [ ]:
# Method 1: Country Rankings
# Walks the leaderboard API page-by-page for each country code,
# stopping once global rank exceeds top_cap.

country_codes = [
    'US', 'RU', 'PL', 'CA', 'DE', 'GB', 'KR', 'BR', 'AU', 'FR', 'JP', 'PH', 'FI', 'ID', 'CL', 'HK', 'CN', 'UA', 'TW', 'SE', 'AR', 'IT', 'TH', 'NL', 'VN', 'ES', 'SG', 'NO', 'MY', 'MX', 'TR', 'NZ', 'RO', 'CZ', 'KZ', 'HU', 'AT', 'DK', 'LT', 'IL', 'PT', 'BY', 'BE', 'PE', 'EE', 'GR', 'SK', 'LV', 'CH', 'BG', 'CO', 'IE', 'RS', 'SA', 'AE', 'IN', 'UY', 'HR', 'ZA', 'VE', 'SI', 'EC', 'EG', 'MA', 'KH', 'DO', 'MD', 'MN', 'PA', 'CR', 'TN', 'KW', 'DZ', 'PR', 'GE', 'PK', 'QA', 'BN', 'MO', 'UZ', 'TT', 'NP', 'PY', 'RE', 'MM', 'KG', 'JO', 'MV', 'MK', 'BA', 'BD', 'GT', 'SV', 'LU', 'BO', 'HN', 'CY', 'IQ', 'BH', 'IS', 'GU', 'NI', 'LB', 'AZ', 'JM', 'MT', 'IR', 'OM', 'LK', 'LA', 'AM', 'GP', 'AL', 'PS', 'MQ', 'MU', 'MP', 'PF', 'NG', 'ME', 'KE', 'BZ', 'MG', 'NC', 'SY', 'LY', 'BB', 'SR', 'GF', 'BS', 'IM', 'GY', 'FO', 'JE', 'GH', 'AW', 'NA', 'GL', 'CI', 'BM', 'SD', 'CU', 'BW', 'LC', 'CW', 'XK', 'AD', 'GG', 'AO', 'GI', 'ET', 'AX', 'YT', 'MZ', 'KY', 'SN', 'CV', 'VI', 'LI', 'AG', 'FJ', 'ZW', 'GA', 'TZ', 'VC', 'ZM', 'UG', 'TG', 'HT', 'YE', 'MF', 'SC', 'KN', 'TJ', 'PM', 'DM', 'SM', 'GD', 'TC', 'RW', 'MC', 'BL', 'CM', 'BT', 'TM', 'AI', 'AP', 'SX', 'PG', 'CG', 'NE', 'TL', 'FM', 'SZ', 'BQ', 'BF', 'DJ', 'MW', 'WS', 'GN', 'BJ', 'SO', 'EU', 'CK', 'CX', 'VG', 'CD', 'SL', 'AF', 'MR', 'ST', 'AS', 'VU', 'ML', 'AQ', 'PW', 'GM', 'LS', 'TD', 'SB', 'SS', 'VA', 'TO', 'KM', 'MH', 'GQ', 'BI', 'NU', 'UM'
]

def _fetch_one_country(country_code: str) -> list[UserCompact]:
    users, cursor = [], None

    while True:
        resp = retry_api_call(
            api.ranking, mode_chosen,
            type=ranking_type, country=country_code, cursor=cursor,
        )

        for entry in resp.ranking:
            user = entry.user
            stats = entry
            if not user or not user.is_active:
                continue
            if not stats.global_rank:
                continue
            rank = stats.global_rank
            if rank_min <= rank <= rank_max:
                users.append(
                    UserCompact(
                        id=entry.user.id,
                        country=country_code,
                        global_rank=rank,
                        pp=stats.pp,
                    )
                )

        # safe last rank check
        last_entry = resp.ranking[-1]
        last_rank = getattr(last_entry, "global_rank", None)

        if last_rank and last_rank > rank_max:
            break

        cursor = resp.cursor

    return users



country_users: list[UserCompact] = []
for i, code in enumerate(country_codes, 1):
    batch = _fetch_one_country(code)
    country_users.extend(batch)
    output.clear()
    print(f"[{i}/{len(country_codes)}] {code}: {len(batch)} users  |  total: {len(country_users)}")

save_users("country_lookup", country_users)
print(f"\n✓ Saved {len(country_users)} users")

Method 2 — Kaggle dataset Cell

In [ ]:
# Method 2: CSV / Kaggle Dataset
# Downloads the community osu! player dataset from Kaggle and converts it
# to UserCompact objects. Requires kagglehub to be authenticated.

import pandas as pd
import kagglehub

dataset_path = kagglehub.dataset_download("ellimaaac/osustandard-players-from-2007-to-today")
csv_files = glob.glob(os.path.join(dataset_path, "*.csv"))
print("Found:", csv_files)

def _csv_to_users(csv_path: str) -> list[UserCompact]:
    df = pd.read_csv(csv_path, sep=';', on_bad_lines='warn', low_memory=False)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    column_mapping = {
        "id": "user_id",
        "country": "country_code",
        "global_ranking": "global_rank",
        "pp": "pp",
    }

    required = set(column_mapping.keys())
    if not required.issubset(df.columns):
        print(f"  Skipping {csv_path}: missing columns {required - set(df.columns)}")
        return []
    df = df.replace("-", pd.NA)
    df = df.dropna(subset=required)

    users = []
    for row in df.itertuples(index=False):
        try:
            users.append(
                UserCompact(
                    id=int(getattr(row, "id")),
                    country=getattr(row, "country"),
                    global_rank=int(getattr(row, "global_ranking")),
                    pp=float(getattr(row, "pp")),
                )
            )
        except ValueError as e:
            print(f"  Skipping row due to conversion error: {e}")
            continue

    return users

csv_users: list[UserCompact] = []
for f in csv_files:
    batch = _csv_to_users(f)
    csv_users.extend(batch)
    print(f"  {os.path.basename(f)}: {len(batch)} rows")

save_users("csv_lookup", csv_users)
print(f"\n✓ Saved {len(csv_users)} users")

Method 3 — Raw ID scan Cell (worse option)

In [ ]:
# Method 3: Raw ID Scan
# Walks sequential user ID blocks across [SCAN_START, SCAN_END].
# Resumes automatically via a progress file if interrupted.

# ID window used by the raw-scan method
SCAN_START = 0           # @param {type:"integer"}
SCAN_END   = 40_000_000  # @param {type:"integer"}

SCAN_PROGRESS_FILE = "raw_progress"
LOG_EVERY          = 400


def _fetch_active_batch(start: int) -> list[UserCompact]:
    """Fetch all active osu! players in a SCAN_BATCH_SIZE-wide ID block."""
    raw = retry_api_call(api.users, list(range(start, start + SCAN_BATCH_SIZE)))
    out = []
    for u in raw:
        stats = getattr(getattr(u, "statistics_rulesets", None), "osu", None)
        if u.is_active and stats and stats.global_rank:
            out.append(
                UserCompact(
                    id=u.id,
                    country=u.country_code,
                    global_rank=stats.global_rank,
                    pp=stats.pp,
                )
            )
    return out


current = load_progress(SCAN_PROGRESS_FILE, default=SCAN_START)
print(f"Resuming from ID {current}")

while current < SCAN_END:
    if current % LOG_EVERY == 0:
        output.clear()

    try:
        batch = retry_api_call(_fetch_active_batch, current)
        save_users(str(current), batch)
        if batch:
            print(f"  ID {current}–{current + SCAN_BATCH_SIZE}: {len(batch)} active users")
        save_progress(SCAN_PROGRESS_FILE, current)

        # --- RANDOM DELAY TO AVOID RATE LIMIT ---
        delay = random.uniform(0.2, 3.0)
        time.sleep(delay)
    except Exception as e:
        print(f"  Error at {current}: {e}")
        time.sleep(3)
        continue

    current += SCAN_BATCH_SIZE

save_progress(SCAN_PROGRESS_FILE, current)
print("✓ Scan complete")

## Merge & Deduplicate Users


In [ ]:
# Merge & Deduplicate Users
# Reads every .jsonl in RESULT_DIR_USERS regardless of which acquisition
# method produced it, then deduplicates by user ID.

def load_all_saved_users() -> list[UserCompact]:
    paths = glob.glob(os.path.join(RESULT_DIR_USERS, "*.jsonl"))
    if not paths:
        print("No user files found in:", RESULT_DIR_USERS)
        return []

    users, errors = [], 0
    for path in paths:
        fname = os.path.basename(path)
        try:
            with open(path) as f:
                for idx, line in enumerate(f):
                    try:
                        d = json.loads(line)
                        users.append(
                            UserCompact(
                                id=int(d["id"]),
                                country=d.get("country"),
                                global_rank=int(d["global_rank"]) if d.get("global_rank") is not None else None,
                                pp=float(d["pp"]) if d.get("pp") is not None else None,
                            )
                        )
                    except Exception as e:
                        errors += 1
                        if errors <= 5:
                            print(f"  ⚠ Parse error in {fname} line {idx}: {e}")
        except Exception as e:
            print(f"  ⚠ File error {fname}: {e}")

    print(f"Raw loaded : {len(users):,} entries from {len(paths)} file(s)")
    if errors:
        print(f"Parse errors: {errors}")
    return users


def deduplicate_users(users: list[UserCompact]) -> list[UserCompact]:
    """Keeps the entry with the highest pp when duplicate IDs are found."""
    best: dict[int, UserCompact] = {}
    for u in users:
        existing = best.get(u.id)
        if existing is None or (u.pp or 0) > (existing.pp or 0):
            best[u.id] = u
    return list(best.values())


raw_users = load_all_saved_users()
users     = deduplicate_users(raw_users)

print(f"After dedup: {len(users):,} unique users")

## Filter & Sort

In [ ]:
# Filter & Sort
# Applies the rank window from config, then sorts by rank ascending.

def filter_users(
    users: list[UserCompact],
    rank_min: int,
    rank_max: int,
) -> list[UserCompact]:
    return [
        u for u in users
        if u.global_rank is not None
        and rank_min <= u.global_rank <= rank_max
    ]

def sort_users(users: list[UserCompact]) -> list[UserCompact]:
    return sorted(users, key=lambda u: u.global_rank)

users    = filter_users(users, rank_min, rank_max)
users    = sort_users(users)
window   = rank_max - rank_min
coverage = len(users) / window if window else 0

print(f"Filtered : {len(users):,} users  (rank {rank_min:,} – {rank_max:,})")
print(f"Coverage : {coverage:.1%}  ({len(users):,} / {window:,} possible ranks)")

## Partition & Save

In [ ]:
# Partition & Save
# Splits the sorted user list into num_parts equal slices for parallel score
# fetching, then saves both the full list and the partition index.

def partition(items: list, n: int) -> list[list]:
    """Divide a list into n roughly equal parts."""
    q, r = divmod(len(items), n)
    parts, start = [], 0
    for i in range(n):
        end = start + q + (1 if i < r else 0)
        parts.append(items[start:end])
        start = end
    return parts


user_parts = partition(users, num_parts)

merged_path = os.path.join(SAVE_DIR, "unique_user_ids.jsonl")
with open(merged_path, "w") as f:
    for u in users:
        f.write(json.dumps(asdict(u)) + "\n")

index_path = os.path.join(SAVE_DIR, "user_parts_index.json")
with open(index_path, "w") as f:
    json.dump(
        {str(i): [u.id for u in part] for i, part in enumerate(user_parts)},
        f,
    )

print(f"✓ Saved {len(users):,} users → {merged_path}")
print(f"✓ Partition index → {index_path}")
print()
for i, part in enumerate(user_parts):
    r = f"{part[0].global_rank:,} – {part[-1].global_rank:,}" if part else "empty"
    print(f"  Part {i:>2}: {len(part):>5,} users  rank {r}")

# 3. Look Up User Score Data

## User Reload

In [ ]:
# Reload users from disk if not already in memory (e.g. after runtime restart).
if "users" not in dir() or not users:
    src = os.path.join(SAVE_DIR, "unique_user_ids.jsonl")
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"User list not found at {src}\n"
            "Run the Merge & Deduplicate cell in Part 2 first."
        )
    users = []
    with open(src) as f:
        for line in f:
            d = json.loads(line)
            users.append(
                UserCompact(
                    id=int(d["id"]),
                    country=d.get("country"),
                    global_rank=int(d["global_rank"]) if d.get("global_rank") is not None else None,
                    pp=float(d["pp"]) if d.get("pp") is not None else None,
                )
            )
    print(f"↺ Restored {len(users):,} users from disk")
else:
    print(f"✓ Users already in memory: {len(users):,}")

## Batch Processing

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

SCORES_PROGRESS = "scores_progress.txt"


def _process_one(user: UserCompact, global_idx: int) -> tuple[list[ScoreCompact], Optional[int]]:
    """Worker executed in thread pool. Returns (scores, error_user_id|None)."""
    try:
        scores = fetch_user_scores(user)
        print(f"  {global_idx:>6,}. {user.id:>10,} ✓  {len(scores)} scores")
        return scores, None
    except Exception as e:
        print(f"  {global_idx:>6,}. {user.id:>10,} ✗  {str(e)[:60]}")
        return [], user.id


def save_score_batch(batch_index: int, scores: list[ScoreCompact]) -> None:
    if not scores:
        return
    path = os.path.join(RESULT_DIR_SCORES, f"batch_{batch_index:04d}.jsonl")
    with open(path, "w") as f:
        for s in scores:
            f.write(json.dumps(asdict(s)) + "\n")


def process_all_users(users: list[UserCompact]) -> None:
    def make_batches(items, size):
        for i in range(0, len(items), size):
            yield i, items[i : i + size]

    start_batch  = load_progress(SCORES_PROGRESS, default=0)
    all_batches  = list(make_batches(users, SCORE_BATCH_SIZE))
    failed_users = []

    print(f"Batches total : {len(all_batches)}")
    print(f"Resuming from : batch {start_batch}")

    for batch_idx, batch in all_batches[start_batch:]:
        output.clear()
        print(f"\nBatch {batch_idx + 1} / {len(all_batches)}  ({len(batch)} users)")

        batch_scores = []
        base_idx     = start_batch * SCORE_BATCH_SIZE + batch_idx * SCORE_BATCH_SIZE

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {
                pool.submit(_process_one, user, base_idx + i): user
                for i, user in enumerate(batch)
            }
            for future in as_completed(futures):
                scores, failed_id = future.result()
                batch_scores.extend(scores)
                if failed_id is not None:
                    failed_users.append(failed_id)

        save_score_batch(batch_idx, batch_scores)
        save_progress(SCORES_PROGRESS, batch_idx + 1)
        print(f"✓ Saved batch {batch_idx + 1}  ({len(batch_scores)} scores)")

    if failed_users:
        fail_path = os.path.join(SAVE_DIR, "failed_users.json")
        with open(fail_path, "w") as f:
            json.dump(failed_users, f)
        print(f"\n⚠ {len(failed_users)} users failed → {fail_path}")

    print("\n✓ All batches complete")

process_all_users(users)

## Retry Failed Users

In [ ]:
# Retry Failed Users
# Re-runs any user IDs written to failed_users.json in the previous step.
# Safe to skip if the file doesn't exist.

fail_path = os.path.join(SAVE_DIR, "failed_users.json")

if not os.path.exists(fail_path):
    print("No failed users to retry.")
else:
    with open(fail_path) as f:
        failed_ids = set(json.load(f))

    retry_users = [u for u in users if u.id in failed_ids]
    print(f"Retrying {len(retry_users)} users")

    retry_scores, still_failed = [], []
    for u in tqdm(retry_users, desc="Retrying"):
        scores, err_id = _process_one(u, u.id)
        retry_scores.extend(scores)
        if err_id:
            still_failed.append(err_id)

    save_batch("retry", retry_scores)

    if still_failed:
        with open(fail_path, "w") as f:
            json.dump(still_failed, f)
        print(f"⚠ Still failing: {len(still_failed)} users")
    else:
        os.remove(fail_path)
        print("✓ All retried successfully")

## scores reload

In [ ]:
# Restore Scores (resume guard)
# Skips reload if scores are already in memory.

if "scores" not in dir() or not scores:
    src = os.path.join(RESULT_DIR_SCORES, "unique_scores.jsonl")
    if not os.path.exists(src):
        raise FileNotFoundError(
            f"Merged score file not found at {src}\n"
            "Run the Merge Scores cell first."
        )

    scores = []
    with open(src, encoding="utf-8") as f:
        for idx, line in enumerate(f):
            try:
                d = json.loads(line)
                scores.append(ScoreCompact(**d))
            except Exception as e:
                print(f"  ⚠ Line {idx}: {e}")

    print(f"↺ Restored {len(scores):,} scores from disk")
else:
    print(f"✓ Scores already in memory: {len(scores):,}")

## merge scores

In [ ]:
# Merge Scores
# Reads every batch_XXXX.jsonl written by the score fetcher, deduplicates by
# score ID, and applies the pp floor. Writes a single flat JSONL as output.

def _parse_bool(val) -> bool:
    if isinstance(val, bool):
        return val
    if isinstance(val, str):
        return val.lower() in ("true", "1", "yes")
    return bool(val)


def load_all_saved_scores(pp_floor: float) -> list[ScoreCompact]:
    paths  = sorted(glob.glob(os.path.join(RESULT_DIR_SCORES, "*.jsonl")))
    if not paths:
        print("No score files found in:", RESULT_DIR_SCORES)
        return []

    seen:   dict[int, ScoreCompact] = {}
    errors  = 0

    for path in paths:
        fname = os.path.basename(path)
        try:
            with open(path, encoding="utf-8") as f:
                for idx, line in enumerate(f, 1):
                    try:
                        d        = json.loads(line)
                        score_id = int(d["id"])
                        if score_id in seen or float(d["pp"]) < pp_floor:
                            continue
                        seen[score_id] = ScoreCompact(
                            id              = score_id,
                            user_id         = int(d["user_id"]),
                            beatmap_id      = int(d["beatmap_id"]),
                            artist          = str(d["artist"]),
                            creator         = str(d["creator"]),
                            title           = str(d["title"]),
                            difficulty_name = str(d["difficulty_name"]),
                            mods            = str(d["mods"]),
                            pp              = float(d["pp"]),
                            accuracy        = float(d["accuracy"]),
                            combo           = int(d["combo"]),
                            rating          = float(d["rating"]),
                            bpm             = float(d["bpm"]),
                            length          = int(d["length"]),
                            ar              = float(d["ar"]),
                            od              = float(d["od"]),
                            cs              = float(d["cs"]),
                            hp              = float(d["hp"]),
                            spinners        = int(d["spinners"]),
                            year            = int(d["year"]),
                            playcount       = int(d["playcount"]),
                            favourite_count = int(d["favourite_count"]),
                            missless        = _parse_bool(d["missless"]),
                            is_hidden       = _parse_bool(d.get("is_hidden", False)),
                        )
                    except Exception as e:
                        errors += 1
                        if errors <= 5:
                            print(f"  ⚠ {fname} line {idx}: {e}")
        except IOError as e:
            print(f"  ⚠ File error {fname}: {e}")

    if errors:
        print(f"Total parse errors: {errors}")
    print(f"Files   : {len(paths)}")
    print(f"Scores  : {len(seen):,}  (pp ≥ {pp_floor})")
    return list(seen.values())


def save_merged_scores(scores: list[ScoreCompact], name: str = "unique_scores") -> None:
    path = os.path.join(RESULT_DIR_SCORES, f"{name}.jsonl")
    with open(path, "w", encoding="utf-8") as f:
        for s in scores:
            f.write(json.dumps(asdict(s)) + "\n")
    print(f"✓ Saved → {path}")


scores = load_all_saved_scores(pp_floor=min_pp)
save_merged_scores(scores)

## group scores

In [ ]:
def build_beatmap_dataset(scores: list[ScoreCompact]) -> list[BeatmapEntry]:
    groups: dict[str, list[ScoreCompact]] = defaultdict(list)
    for s in scores:
        groups[f"{s.beatmap_id}_{s.mods}"].append(s)

    entries = []
    for items in groups.values():
        first = items[0]
        n     = len(items)
        entries.append(BeatmapEntry(
            beatmap_id      = first.beatmap_id,
            mods            = first.mods,
            artist          = first.artist,
            creator         = first.creator,
            title           = first.title,
            difficulty_name = first.difficulty_name,
            rating          = first.rating,
            bpm             = first.bpm,
            length          = first.length,
            ar              = first.ar,
            od              = first.od,
            cs              = first.cs,
            hp              = first.hp,
            spinners        = first.spinners,
            year            = first.year,
            playcount       = first.playcount,
            favourite_count = first.favourite_count,
            combo           = first.combo,
            pp_avg          = sum(s.pp       for s in items) / n,
            accuracy_avg    = sum(s.accuracy for s in items) / n,
            hidden_rate     = sum(s.is_hidden  for s in items) / n,
            missless_rate   = sum(s.missless   for s in items) / n,
            amount_of_scores= n,
        ))

    return entries


beatmaps = build_beatmap_dataset(scores)
print(f"Groups before filter : {len(beatmaps):,}")

## filter and sort beatmaps

In [ ]:
# Filter & Sort Beatmaps

beatmaps = [b for b in beatmaps if b.amount_of_scores >= min_score_count]
beatmaps.sort(key=lambda b: b.amount_of_scores, reverse=True)

print(f"Groups after filter  : {len(beatmaps):,}  (≥ {min_score_count} scores)")

del scores


_bm_path = os.path.join(RESULT_DIR_SCORES, "beatmaps_filtered.jsonl")
with open(_bm_path, "w", encoding="utf-8") as _f:
    for b in beatmaps:
        _f.write(json.dumps(asdict(b)) + "\n")
print(f"✓ Saved {len(beatmaps):,} beatmaps → {_bm_path}")

## Api Fetch Extra Beatmapset information

In [ ]:
# Beatmapset Cache
# Keyed by "artist|||creator|||title" so beatmaps sharing a set only cost one
# API call. Persisted as JSON (human-readable, no pickle version pinning).

def _cache_key(artist: str, creator: str, title: str) -> str:
    parts = [" ".join(str(p).lower().strip().split()) for p in (artist, creator, title)]
    return "|||".join(parts)


def load_cache() -> dict:
    if not os.path.exists(CACHE_FILE):
        return {}
    try:
        with open(CACHE_FILE, encoding="utf-8") as f:
            data = json.load(f)
        print(f"↺ Cache loaded: {len(data):,} entries")
        return data
    except Exception as e:
        print(f"⚠ Cache load failed: {e} — starting empty")
        return {}


def save_cache(cache: dict) -> None:
    try:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f)
    except Exception as e:
        print(f"⚠ Cache save failed: {e}")


def _top_tags(top_tag_ids, related_tags, n: int) -> list[str]:
    if not top_tag_ids or not related_tags:
        return []
    id_to_name = {t.id: t.name for t in related_tags}

    def _count(t):
        return t["count"] if isinstance(t, dict) else t.count

    def _tag_id(t):
        return t["tag_id"] if isinstance(t, dict) else t.tag_id

    ranked = sorted(top_tag_ids, key=_count, reverse=True)
    return [id_to_name[_tag_id(t)] for t in ranked[:n] if _tag_id(t) in id_to_name]



In [ ]:
# Beatmapset API Fetch
# Pulls genre, language, tags, and ranked_date for one beatmapset.
# Returns a plain dict stored in the cache; never raises — returns a minimal
# fallback so the enhancement loop can continue on API errors.

_EMPTY = {"genre": None, "language": None, "ranked_date": None,
          "tags": [], "beatmaps": {}}


def fetch_beatmapset(beatmap_id: int, artist: str, creator: str,
                     title: str, cache: dict) -> dict:
    key = _cache_key(artist, creator, title)
    if key in cache:
        return cache[key]

    try:
        bs   = retry_api_call(api.beatmapset, beatmap_id=beatmap_id)
        data = {
            "genre":       bs.genre["name"]    if bs.genre    else None,
            "language":    bs.language["name"] if bs.language else None,
            "ranked_date": bs.ranked_date.year if bs.ranked_date else None,
            "tags":        [t.name for t in bs.related_tags] if bs.related_tags else [],
            "beatmaps":    {
                bm.id: _top_tags(
                    getattr(bm, "top_tag_ids", None),
                    bs.related_tags,
                    MAX_TAGS,
                )
                for bm in bs.beatmaps
            },
        }
        cache[key] = data
        return data

    except Exception as e:
        print(f"  ⚠ beatmapset {beatmap_id}: {e}")
        return _EMPTY

In [ ]:
# Enhance Beatmaps
# Iterates over BeatmapEntry list, fetches extra metadata for each unique
# beatmapset, and returns EnhancedBeatmapEntry objects.

def enhance_beatmaps(
    beatmaps: list[BeatmapEntry],
) -> list[EnhancedBeatmapEntry]:

    cache       = load_cache()
    enhanced    = []
    cache_hits  = 0
    errors      = 0
    total       = len(beatmaps)

    for i, b in enumerate(beatmaps):
        key      = _cache_key(b.artist, b.creator, b.title)
        was_hit  = key in cache

        data     = fetch_beatmapset(b.beatmap_id, b.artist, b.creator, b.title, cache)

        if was_hit:
            cache_hits += 1

        # Prefer year already on the entry; fall back to ranked_date from API
        year = b.year or data.get("ranked_date") or 0

        diff_tags = data["beatmaps"].get(b.beatmap_id, [])

        enhanced.append(EnhancedBeatmapEntry(
            **asdict(b),
            genre    = data["genre"],
            language = data["language"],
            tags     = ", ".join(diff_tags) if diff_tags else None,
        ))

        if (i + 1) % CACHE_SAVE_INTERVAL == 0:
            save_cache(cache)
            output.clear()
            print(f"Progress : {i + 1:,} / {total:,}")
            print(f"Cache    : {len(cache):,} entries  ({cache_hits} hits)")

    save_cache(cache)

    misses   = total - cache_hits
    hit_rate = cache_hits / total * 100 if total else 0
    print(f"\n✓ Enhanced {len(enhanced):,} beatmaps")
    print(f"  Cache hits   : {cache_hits:,}  ({hit_rate:.1f}%)")
    print(f"  API calls    : {misses:,}")

    return enhanced


enhanced_beatmaps = enhance_beatmaps(beatmaps)

## Save Beatmapsets

In [ ]:
# Save Enhanced Beatmaps
_out = os.path.join(RESULT_DIR_SCORES, "enhanced_beatmaps.jsonl")
with open(_out, "w", encoding="utf-8") as f:
    for b in enhanced_beatmaps:
        f.write(json.dumps(asdict(b)) + "\n")

print(f"✓ Saved {len(enhanced_beatmaps):,} → {_out}")

# 4. Export to Excel Cell

In [ ]:
# Export to Excel
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment,
                              Border, Side, GradientFill)
from openpyxl.utils import get_column_letter

# Load beatmaps
if "enhanced_beatmaps" not in dir() or not enhanced_beatmaps:
    src = os.path.join(RESULT_DIR_SCORES, "enhanced_beatmaps.jsonl")
    if not os.path.exists(src):
        raise FileNotFoundError(f"Run the Enhance Beatmaps cell first.\nExpected: {src}")
    enhanced_beatmaps = []
    with open(src, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            enhanced_beatmaps.append(EnhancedBeatmapEntry(**d))
    print(f"↺ Restored {len(enhanced_beatmaps):,} enhanced beatmaps")


# Helpers

def _fmt_time(seconds: int) -> str:
    return f"{seconds // 60}:{seconds % 60:02d}"

def _pct(val: float) -> str:
    return f"{val * 100:.2f}%"

def _url(beatmap_id: int) -> str:
    return f"https://osu.ppy.sh/b/{beatmap_id}"


# ── Style constants

COL_HEADER_FILL  = "1A1A2E"   # deep navy
COL_HEADER_FONT  = "E0E0FF"   # soft lavender white
ROW_ALT_FILL     = "F4F4FB"   # very light blue-grey for alternating rows
ACCENT_FILL      = "16213E"   # darker navy for frozen pane indicator
HYPERLINK_COLOUR = "4A90D9"

THIN = Side(style="thin", color="D0D0E0")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

COLUMNS = [
    # (header,          attr/lambda,                         width, number_format)
    ("Title",           lambda b: b.title,                   28,    None),
    ("Diff",            lambda b: b.difficulty_name,         22,    None),
    ("Creator",         lambda b: b.creator,                 18,    None),
    ("Artist",          lambda b: b.artist,                  22,    None),
    ("Year",            lambda b: b.year or "",              7,     None),
    ("Genre",           lambda b: b.genre or "",             14,    None),
    ("Language",        lambda b: b.language or "",          13,    None),
    ("Stars",           lambda b: round(b.rating, 2),        8,     "0.00"),
    ("Length",          lambda b: _fmt_time(b.length),       9,     None),
    ("BPM",             lambda b: round(b.bpm, 1),           7,     "0.0"),
    ("Mods",            lambda b: b.mods,                    8,     None),
    ("AR",              lambda b: round(b.ar, 1),            6,     "0.0"),
    ("OD",              lambda b: round(b.od, 1),            6,     "0.0"),
    ("HP",              lambda b: round(b.hp, 1),            6,     "0.0"),
    ("CS",              lambda b: round(b.cs, 1),            6,     "0.0"),
    ("Spins",           lambda b: b.spinners,                7,     None),
    ("Max Combo",       lambda b: b.combo,                   11,    "#,##0"),
    ("Avg PP",          lambda b: round(b.pp_avg, 1),        9,     "0.0"),
    ("HD Rate",         lambda b: _pct(b.hidden_rate),       9,     None),
    ("Avg Acc",         lambda b: _pct(b.accuracy_avg),      9,     None),
    ("Missless Rate",   lambda b: _pct(b.missless_rate),     13,    None),
    ("Playcount",       lambda b: b.playcount,               12,    "#,##0"),
    ("Favs",            lambda b: b.favourite_count,         9,     "#,##0"),
    ("Tags",            lambda b: b.tags or "",              40,    None),
    ("#",               lambda b: b.amount_of_scores,        7,     "#,##0"),
    ("URL",             lambda b: _url(b.beatmap_id),        38,    None),
]


# ── Build workbook ────────────────────────────────────────────────────────────

wb = Workbook()
ws = wb.active
ws.title = "Beatmaps"

header_font    = Font(name="Calibri", bold=True, color=COL_HEADER_FONT, size=10)
header_fill    = PatternFill("solid", fgColor=COL_HEADER_FILL)
header_align   = Alignment(horizontal="center", vertical="center", wrap_text=False)
alt_fill       = PatternFill("solid", fgColor=ROW_ALT_FILL)
cell_font      = Font(name="Calibri", size=10)
center_align   = Alignment(horizontal="center", vertical="center")
left_align     = Alignment(horizontal="left",   vertical="center")
hyperlink_font = Font(name="Calibri", size=10, color=HYPERLINK_COLOUR, underline="single")

# Header row
for col_idx, (header, _, width, _fmt) in enumerate(COLUMNS, start=1):
    cell = ws.cell(row=1, column=col_idx, value=header)
    cell.font      = header_font
    cell.fill      = header_fill
    cell.alignment = header_align
    cell.border    = BORDER
    ws.column_dimensions[get_column_letter(col_idx)].width = width

ws.row_dimensions[1].height = 22

# Data rows
url_col = next(i + 1 for i, (h, *_) in enumerate(COLUMNS) if h == "URL")

for row_idx, b in enumerate(enhanced_beatmaps, start=2):
    is_alt = row_idx % 2 == 0
    for col_idx, (header, extractor, _, fmt) in enumerate(COLUMNS, start=1):
        value = extractor(b)
        cell  = ws.cell(row=row_idx, column=col_idx, value=value)
        cell.border = BORDER
        cell.font   = cell_font

        if col_idx == url_col:
            cell.hyperlink = value
            cell.font      = hyperlink_font
            cell.alignment = left_align
        elif col_idx in (1, 2, 3, 4, 6, 7, 24):   # text columns — left align
            cell.alignment = left_align
        else:
            cell.alignment = center_align

        if fmt:
            cell.number_format = fmt
        if is_alt and col_idx != url_col:
            cell.fill = alt_fill

ws.freeze_panes = "A2"
ws.auto_filter.ref = f"A1:{get_column_letter(len(COLUMNS))}1"


# ── Save ──────────────────────────────────────────────────────────────────────

out_path = os.path.join(SAVE_DIR, f"{OUTPUT_FILENAME}.xlsx")
wb.save(out_path)

# Download directly to local machine
files.download(out_path)
print(f"✓ Saved & downloading → {out_path}")
print(f"  {len(enhanced_beatmaps):,} rows  ×  {len(COLUMNS)} columns")